In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Runtime 작업 디버그

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
리소스를 많이 사용하는 Qiskit Runtime 워크로드를 하드웨어에서 실행하기 위해 제출하기 전에, Qiskit Runtime [`Neat` (Noisy Estimator Analyzer Tool)](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/debug-tools-neat#neat) 클래스를 사용하여 Estimator 워크로드가 올바르게 설정되어 있는지, 정확한 결과를 반환할 가능성이 높은지, 지정된 문제에 가장 적합한 옵션을 사용하는지 등을 확인할 수 있습니다.

`Neat`는 효율적인 시뮬레이션을 위해 입력 Circuit을 클리퍼드화(Cliffordize)하면서 Circuit의 구조와 깊이를 그대로 유지합니다. 클리퍼드 Circuit은 유사한 수준의 노이즈를 겪으며, 관심 있는 원래 Circuit을 연구하기 위한 좋은 대용품이 됩니다.

다음 예제들은 `Neat`가 유용한 자원이 될 수 있는 상황을 보여 줍니다.
먼저, 관련 패키지를 가져오고 [Qiskit Runtime 서비스에 인증합니다.](/guides/cloud-setup)
## 환경 준비

In [1]:
import numpy as np
import random

from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
from qiskit_ibm_runtime.debug_tools import Neat

from qiskit_aer.noise import NoiseModel, depolarizing_error

In [2]:
# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an equivalent Instruction Set Architecture (ISA) circuit.
pm = generate_preset_pass_manager(backend=backend, optimization_level=0)

# Set the random seed
random.seed(10)

## 대상 Circuit 초기화
다음 특성을 가진 6-Qubit Circuit을 고려합니다.

* 무작위 `RZ` 회전과 `CNOT` Gate 레이어를 번갈아 적용합니다.
* 미러 구조를 가지고 있습니다. 즉, 유니터리 `U`를 적용한 후 그 역을 적용합니다.

In [3]:
def generate_circuit(n_qubits, n_layers):
    r"""
    A function to generate a pseudo-random a circuit with ``n_qubits`` qubits and
    ``2*n_layers`` entangling layers of the type used in this notebook.
    """
    # An array of random angles
    angles = [
        [random.random() for q in range(n_qubits)] for s in range(n_layers)
    ]

    qc = QuantumCircuit(n_qubits)
    qubits = list(range(n_qubits))

    # do random circuit
    for layer in range(n_layers):
        # rotations
        for q_idx, qubit in enumerate(qubits):
            qc.rz(angles[layer][q_idx], qubit)

        # cx gates
        control_qubits = (
            qubits[::2] if layer % 2 == 0 else qubits[1 : n_qubits - 1 : 2]
        )
        for qubit in control_qubits:
            qc.cx(qubit, qubit + 1)

    # undo random circuit
    for layer in range(n_layers)[::-1]:
        # cx gates
        control_qubits = (
            qubits[::2] if layer % 2 == 0 else qubits[1 : n_qubits - 1 : 2]
        )
        for qubit in control_qubits:
            qc.cx(qubit, qubit + 1)

        # rotations
        for q_idx, qubit in enumerate(qubits):
            qc.rz(-angles[layer][q_idx], qubit)

    return qc


# Generate a random circuit
qc = generate_circuit(6, 3)
# Convert the abstract circuit to an equivalent ISA circuit.
isa_qc = pm.run(qc)

qc.draw("mpl", idle_wires=0)

<Image src="../docs/images/guides/debug-qiskit-runtime-jobs/extracted-outputs/df19af55-897d-4b1f-baf8-fac2641ae87d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/debug-qiskit-runtime-jobs/extracted-outputs/df19af55-897d-4b1f-baf8-fac2641ae87d-0.svg)

단일 파울리 `Z` 연산자를 관측값으로 선택하고, 이를 사용하여 primitive unified bloc(PUB)을 초기화합니다.

In [4]:
# Initialize the observables
obs = ["ZIIIII", "IZIIII", "IIZIII", "IIIZII", "IIIIZI", "IIIIIZ"]
print(f"Observables: {obs}")

# Map the observables to the backend's layout
isa_obs = [SparsePauliOp(o).apply_layout(isa_qc.layout) for o in obs]

# Initialize the PUBs, which consist of six-qubit circuits with `n_layers` 1, ..., 6
all_n_layers = [1, 2, 3, 4, 5, 6]

pubs = [(pm.run(generate_circuit(6, n)), isa_obs) for n in all_n_layers]

Observables: ['ZIIIII', 'IZIIII', 'IIZIII', 'IIIZII', 'IIIIZI', 'IIIIIZ']


## Cliffordize the circuits

The previously defined PUB circuits are not Clifford, which makes them difficult to simulate classically. However, you can use the `Neat` [`to_clifford`](/docs/api/qiskit-ibm-runtime/debug-tools-neat#to_clifford) method to map them to Clifford circuits for more efficient simulation.  The [`to_clifford`](/docs/api/qiskit-ibm-runtime/debug-tools-neat#to_clifford) method is a wrapper around the [`ConvertISAToClifford`](/docs/api/qiskit-ibm-runtime/transpiler-passes-convert-isa-to-clifford) transpiler pass, which can also be used independently. In particular, it replaces non-Clifford single-qubit gates in the original circuit with Clifford single-qubit gates, but it does not mutate the two-qubit gates, number of qubits, or circuit depth.

See [Efficient simulation of stabilizer circuits with Qiskit Aer primitives](/docs/guides/simulate-stabilizer-circuits) for more information about Clifford circuit simulation.

First, initialize `Neat`.

In [5]:
# You could specify a custom `NoiseModel` here. If `None`, `Neat`
# pulls the noise model from the given backend
noise_model = None

# Initialize `Neat`
analyzer = Neat(backend, noise_model)

## Circuit 클리퍼드화
앞서 정의한 PUB Circuit은 클리퍼드가 아니기 때문에 고전적으로 시뮬레이션하기 어렵습니다. 그러나 `Neat`의 [`to_clifford`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/debug-tools-neat#to_clifford) 메서드를 사용하여 더 효율적인 시뮬레이션을 위해 클리퍼드 Circuit으로 매핑할 수 있습니다. [`to_clifford`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/debug-tools-neat#to_clifford) 메서드는 [`ConvertISAToClifford`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/transpiler-passes-convert-isa-to-clifford) Transpiler 패스의 래퍼로, 독립적으로도 사용할 수 있습니다. 특히, 원래 Circuit의 비-클리퍼드 단일-Qubit Gate를 클리퍼드 단일-Qubit Gate로 대체하지만, 2-Qubit Gate, Qubit 수, Circuit 깊이는 변경하지 않습니다.

클리퍼드 Circuit 시뮬레이션에 대한 자세한 내용은 [Qiskit Aer 프리미티브를 사용한 안정자 Circuit의 효율적인 시뮬레이션](/guides/simulate-stabilizer-circuits)을 참조하세요.
먼저, `Neat`를 초기화합니다.

In [6]:
clifford_pubs = analyzer.to_clifford(pubs)

clifford_pubs[0].circuit.draw("mpl", idle_wires=0)

<Image src="../docs/images/guides/debug-qiskit-runtime-jobs/extracted-outputs/3ad78f41-a2f8-4381-826a-ae728e081ad6-0.svg" alt="Output of the previous code cell" />

다음으로, PUB를 클리퍼드화합니다.

In [7]:
# Perform a noiseless simulation
ideal_results = analyzer.ideal_sim(clifford_pubs)
print(f"Ideal results:\n {ideal_results}\n")

# Perform a noisy simulation with the backend's noise model
noisy_results = analyzer.noisy_sim(clifford_pubs)
print(f"Noisy results:\n {noisy_results}\n")

Ideal results:
 NeatResult([NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.])), NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.])), NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.])), NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.])), NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.])), NeatPubResult(vals=array([1., 1., 1., 1., 1., 1.]))])



Noisy results:
 NeatResult([NeatPubResult(vals=array([0.99023438, 0.99609375, 0.9921875 , 0.99023438, 0.99414062,
       0.99414062])), NeatPubResult(vals=array([0.984375  , 0.99414062, 0.98242188, 0.98828125, 0.98632812,
       0.99414062])), NeatPubResult(vals=array([0.96679688, 0.97070312, 0.95898438, 0.97851562, 0.98046875,
       0.98828125])), NeatPubResult(vals=array([0.9453125 , 0.953125  , 0.97070312, 0.96875   , 0.98242188,
       0.99023438])), NeatPubResult(vals=array([0.93164062, 0.9375    , 0.953125  , 0.96875   , 0.96484375,
       0.98046875])), NeatPubResult(vals=array([0.92578125, 0.921875  , 0.93359375, 0.953125  , 0.95898438,
       0.9765625 ]))])



![Output of the previous code cell](../docs/images/guides/debug-qiskit-runtime-jobs/extracted-outputs/3ad78f41-a2f8-4381-826a-ae728e081ad6-0.svg)
## 애플리케이션 1: Circuit 출력에 대한 노이즈의 영향 분석
이 예제는 `Neat`를 사용하여 이상적인([`ideal_sim`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/debug-tools-neat#ideal_sim)) 조건과 노이즈가 있는([`noisy_sim`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/debug-tools-neat#noisy_sim)) 조건에서 시뮬레이션을 실행함으로써, Circuit 깊이의 함수로서 다양한 노이즈 모델이 PUB에 미치는 영향을 연구하는 방법을 보여줍니다. 이는 QPU에서 작업을 실행하기 전에 실험 결과의 품질에 대한 기대치를 설정하는 데 유용합니다. 노이즈 모델에 대한 자세한 내용은 [Qiskit Aer 프리미티브를 이용한 정확한 시뮬레이션 및 노이즈 시뮬레이션](/guides/simulate-with-qiskit-aer#exact-and-noisy-simulation-with-qiskit-aer-primitives)을 참조하세요.

시뮬레이션 결과는 수학적 연산을 지원하므로, 서로 비교하거나 (또는 실험 결과와 비교하여) 성능 지표를 계산하는 데 활용할 수 있습니다.

> **Caution:** QPU는 다양한 종류의 노이즈에 영향을 받을 수 있습니다. 여기서 사용된 Qiskit Aer 노이즈 모델은 그 중 일부만 시뮬레이션하므로, 실제 QPU의 노이즈보다 덜 심각할 가능성이 높습니다.
> 
> QPU에서 노이즈 모델을 초기화할 때 포함되는 오류에 대한 자세한 내용은 Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) API 레퍼런스를 참조하세요.

먼저 이상적인 조건과 노이즈가 있는 조건에서의 고전적 시뮬레이션을 수행합니다.

In [8]:
# Figure of merit: Absolute difference
def rdiff(res1, re2):
    r"""The absolute difference between `res1` and re2`.

    --> The closer to `0`, the better.
    """
    d = abs(res1 - re2)
    return np.round(d.vals * 100, 2)


for idx, (ideal_res, noisy_res) in enumerate(
    zip(ideal_results, noisy_results)
):
    vals = rdiff(ideal_res, noisy_res)

    # Print the mean absolute difference for the observables
    mean_vals = np.round(np.mean(vals), 2)
    print(
        f"Mean absolute difference between ideal and noisy results for circuits with {all_n_layers[idx]} layers:\n  {mean_vals}%\n"
    )

Mean absolute difference between ideal and noisy results for circuits with 1 layers:
  0.72%

Mean absolute difference between ideal and noisy results for circuits with 2 layers:
  1.17%

Mean absolute difference between ideal and noisy results for circuits with 3 layers:
  2.6%

Mean absolute difference between ideal and noisy results for circuits with 4 layers:
  3.16%

Mean absolute difference between ideal and noisy results for circuits with 5 layers:
  4.4%

Mean absolute difference between ideal and noisy results for circuits with 6 layers:
  5.5%



You can follow these rough and simplified guidelines to improve circuits of this type:

- If the mean absolute difference is greater than 90%, mitigation will likely not help.
- If the mean absolute difference is less than 90%, [Probabilistic Error Amplification (PEA)](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) will likely be able to improve the results.
- If the mean absolute difference is less than 80%, [ZNE with gate folding](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) will also likely be able to improve the results.

Because all of the absolute differences above are less than 90%, applying PEA to the original circuit will hopefully improve the quality of its results.

You can specify different noise models in the analyzer. The following example performs the same test but adds a custom noise model.

In [9]:
# Set up a noise model with strength 0.02 on every two-qubit gate
noise_model = NoiseModel()
for qubits in backend.coupling_map:
    noise_model.add_quantum_error(
        depolarizing_error(0.02, 2), ["ecr", "cx"], qubits
    )

# Update the analyzer's noise model
analyzer.noise_model = noise_model

# Perform a noiseless simulation
ideal_results = analyzer.ideal_sim(clifford_pubs)

# Perform a noisy simulation with the backend's noise model
noisy_results = analyzer.noisy_sim(clifford_pubs)

# Compare the results
for idx, (ideal_res, noisy_res) in enumerate(
    zip(ideal_results, noisy_results)
):
    values = rdiff(ideal_res, noisy_res)

    # Print the mean absolute difference for the observables
    mean_values = np.round(np.mean(values), 2)
    print(
        f"Mean absolute difference between ideal and noisy results for circuits with {all_n_layers[idx]} layers:\n  {mean_values}%\n"
    )

Mean absolute difference between ideal and noisy results for circuits with 1 layers:
  0.0%

Mean absolute difference between ideal and noisy results for circuits with 2 layers:
  0.0%

Mean absolute difference between ideal and noisy results for circuits with 3 layers:
  0.0%

Mean absolute difference between ideal and noisy results for circuits with 4 layers:
  0.0%

Mean absolute difference between ideal and noisy results for circuits with 5 layers:
  0.0%

Mean absolute difference between ideal and noisy results for circuits with 6 layers:
  0.0%



As shown, given a noise model, you can try to quantify the impact of noise on the (Cliffordized version of the) PUBs of interest before running them on a QPU.

## Application 2: Benchmark different strategies

This example uses `Neat` to help identify the best options for your PUBs. To do so, consider running an estimation problem with PEA, which cannot be simulated with `qiskit_aer`. You can use `Neat` to help determine which noise amplification factors will work best, then use those factors when running the original experiment on a QPU.

In [10]:
# Generate a circuit with six qubits and six layers
isa_qc = pm.run(generate_circuit(6, 3))

# Use the same observables as previously
pubs = [(isa_qc, isa_obs)]
clifford_pubs = analyzer.to_clifford(pubs)

In [11]:
noise_factors = [
    [1, 1.1],
    [1, 1.1, 1.2],
    [1, 1.5, 2],
    [1, 1.5, 2, 2.5, 3],
    [1, 4],
]

In [12]:
# Run the PUBs on a QPU
estimator = Estimator(backend)
estimator.options.default_shots = 100000
estimator.options.twirling.enable_gates = True
estimator.options.twirling.enable_measure = True
estimator.options.twirling.shots_per_randomization = 100
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

jobs = []
for factors in noise_factors:
    estimator.options.resilience.zne.noise_factors = factors
    jobs.append(estimator.run(clifford_pubs))

results = [job.result() for job in jobs]

In [13]:
# Perform a noiseless simulation
ideal_results = analyzer.ideal_sim(clifford_pubs)

In [14]:
# Look at the mean absolute difference to quickly tell the best choice for your options
for factors, res in zip(noise_factors, results):
    d = rdiff(ideal_results[0], res[0])
    print(
        f"Mean absolute difference for factors {factors}:\n  {np.round(np.mean(d), 2)}%\n"
    )

Mean absolute difference for factors [1, 1.1]:
  6.83%

Mean absolute difference for factors [1, 1.1, 1.2]:
  8.76%

Mean absolute difference for factors [1, 1.5, 2]:
  8.03%

Mean absolute difference for factors [1, 1.5, 2, 2.5, 3]:
  10.17%

Mean absolute difference for factors [1, 4]:
  8.02%



위에서 보이듯이, 노이즈 모델이 주어지면 관심 있는 PUB(의 클리포드화 버전)을 QPU에서 실행하기 전에 노이즈가 미치는 영향을 정량화하려는 시도를 할 수 있습니다.

## 애플리케이션 2: 다양한 전략 벤치마킹

이 예제는 `Neat`를 사용하여 PUB에 가장 적합한 옵션을 식별하는 방법을 보여줍니다. 이를 위해 `qiskit_aer`로 시뮬레이션할 수 없는 PEA를 사용한 추정 문제를 실행하는 것을 고려합니다. `Neat`를 사용하여 가장 적합한 노이즈 증폭 인수를 결정한 다음, QPU에서 원래 실험을 실행할 때 해당 인수를 사용할 수 있습니다.